# Gold : indicateurs par catégorie

**Entrée :** `workspace.gold.fact_order_items`
**Sorties :**
- `workspace.gold.kpi_category_revenue` : une ligne par catégorie
- `workspace.gold.kpi_monthly_category_revenue` : une ligne par mois et par catégorie

Seules les **ventes** sont comptées (`is_sale` : commande ni `canceled` ni `unavailable`). Le chiffre d'affaires est le prix des articles, sans les frais de port, comme dans `kpi_monthly_revenue`.

| Colonne | Sens |
|---|---|
| `item_count` | nombre d'articles vendus |
| `order_count` | nombre de commandes distinctes contenant la catégorie |
| `product_revenue` | somme des prix |
| `share_pct` | part de la catégorie dans le chiffre d'affaires total (en %) |

In [0]:
CREATE OR REPLACE TABLE workspace.gold.kpi_category_revenue AS
SELECT
    category,
    COUNT(*)                                                  AS item_count,
    COUNT(DISTINCT order_id)                                  AS order_count,
    SUM(price)                                                AS product_revenue,
    SUM(freight_value)                                        AS freight_total,
    ROUND(100.0 * SUM(price) / SUM(SUM(price)) OVER (), 2)    AS share_pct
FROM workspace.gold.fact_order_items
WHERE is_sale
GROUP BY category

num_affected_rows,num_inserted_rows


In [0]:
CREATE OR REPLACE TABLE workspace.gold.kpi_monthly_category_revenue AS
SELECT
    order_month,
    category,
    COUNT(*)          AS item_count,
    SUM(price)        AS product_revenue
FROM workspace.gold.fact_order_items
WHERE is_sale
GROUP BY order_month, category

num_affected_rows,num_inserted_rows


## Vérification
Les trois totaux doivent être **identiques** : 13 494 400,74 (chiffre d'affaires des ventes).

In [0]:
SELECT
    (SELECT SUM(product_revenue) FROM workspace.gold.kpi_category_revenue)          AS ca_par_categorie,
    (SELECT SUM(product_revenue) FROM workspace.gold.kpi_monthly_category_revenue)  AS ca_par_mois_et_categorie,
    (SELECT SUM(product_revenue) FROM workspace.gold.kpi_monthly_revenue)           AS ca_kpi_mensuel,
    (SELECT ROUND(SUM(share_pct), 1) FROM workspace.gold.kpi_category_revenue)      AS somme_parts_pct

ca_par_categorie,ca_par_mois_et_categorie,ca_kpi_mensuel,somme_parts_pct
13494400.74,13494400.74,13494400.74,100.0


## Les 10 premières catégories

In [0]:
SELECT category, item_count, order_count, product_revenue, share_pct
FROM workspace.gold.kpi_category_revenue
ORDER BY product_revenue DESC
LIMIT 10

category,item_count,order_count,product_revenue,share_pct
health_beauty,9634,8800,1255695.13,9.31
watches_gifts,5970,5604,1198185.21,8.88
bed_bath_table,11097,9399,1035964.06,7.68
sports_leisure,8590,7673,979740.92,7.26
computers_accessories,7781,6654,904322.02,6.70
furniture_decor,8298,6425,727465.05,5.39
housewares,6915,5847,626825.80,4.65
cool_stuff,3779,3616,620770.49,4.60
auto,4204,3872,586585.73,4.35
garden_tools,4328,3505,481009.94,3.56
